In [5]:
import os
import re
import json
import PyPDF2

def trim_pdf(input_path, output_path, skip_first=4, skip_last=1):
    with open(input_path, 'rb') as infile:
        reader = PyPDF2.PdfReader(infile)
        writer = PyPDF2.PdfWriter()

        total_pages = len(reader.pages)
        start = skip_first
        end = total_pages - skip_last

        if start >= end:
            print(f"Skipping {input_path}: Not enough pages to trim.")
            return None

        for i in range(start, end):
            writer.add_page(reader.pages[i])

        with open(output_path, 'wb') as outfile:
            writer.write(outfile)

        return output_path  # return for further processing

def find_index_end_page(pdf_path):
    pattern = re.compile(r"\[\s*(19|20|21|22|23|24|25)\d{2}")
    with open(pdf_path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text:
                first_line = text.strip().split("\n")[0]
                if pattern.search(first_line):  # changed from match() to search()
                    return i + 1  # 1-based index
    return None


def process_all_pdfs_in_folder(folder_path='.', skip_first=4, skip_last=2):
    trimmed_folder = os.path.join(folder_path, 'trimmed')
    os.makedirs(trimmed_folder, exist_ok=True)

    index_map = {}

    for filename in os.listdir(folder_path):
        if filename.lower().endswith('.pdf'):
            input_path = os.path.join(folder_path, filename)
            output_path = os.path.join(trimmed_folder, filename)
            print(f"\nProcessing: {filename}")
            
            trimmed = trim_pdf(input_path, output_path, skip_first, skip_last)
            if trimmed:
                page_num = find_index_end_page(trimmed)
                if page_num:
                    print(f"➤ Index ends at page: {page_num}")
                    index_map[filename] = page_num
                else:
                    print("⚠️  Could not find start-of-case pattern.")
                    index_map[filename] = None

                total_pages = len(PyPDF2.PdfReader(open(trimmed, 'rb')).pages)
                print(f"Trimmed PDF has {total_pages} pages.")

    # Save index mapping as JSON
    index_json_path = os.path.join(trimmed_folder, 'index_mapping.json')
    with open(index_json_path, 'w') as f:
        json.dump(index_map, f, indent=4)
    print(f"\n✅ Index mapping saved to {index_json_path}")

# Run it
process_all_pdfs_in_folder()



Processing: 2024-1.pdf
➤ Index ends at page: 5
Trimmed PDF has 1252 pages.

Processing: 2024-10.pdf
➤ Index ends at page: 3
Trimmed PDF has 316 pages.

Processing: 2024-11.pdf
➤ Index ends at page: 3
Trimmed PDF has 326 pages.

Processing: 2024-12.pdf
➤ Index ends at page: 3
Trimmed PDF has 336 pages.

Processing: 2024-2.pdf
➤ Index ends at page: 5
Trimmed PDF has 1244 pages.

Processing: 2024-3.pdf
➤ Index ends at page: 5
Trimmed PDF has 1364 pages.

Processing: 2024-4.pdf
➤ Index ends at page: 5
Trimmed PDF has 778 pages.

Processing: 2024-5.pdf
➤ Index ends at page: 5
Trimmed PDF has 1314 pages.

Processing: 2024-6.pdf
➤ Index ends at page: 5
Trimmed PDF has 966 pages.

Processing: 2024-7.pdf
➤ Index ends at page: 7
Trimmed PDF has 2454 pages.

Processing: 2024-8.pdf
➤ Index ends at page: 3
Trimmed PDF has 291 pages.

Processing: 2024-9.pdf
➤ Index ends at page: 3
Trimmed PDF has 313 pages.

✅ Index mapping saved to .\trimmed\index_mapping.json


In [34]:
import fitz  # PyMuPDF
import re
import json
import os

def extract_index_page_numbers(pdf_folder, pdf_name, json_name="index_mapping.json", debug=False):
    """
    Reads index range from JSON and extracts page numbers from index pages in a PDF.

    Returns:
        Tuple[List[int], int]: (List of adjusted page numbers, last_index_page)
    """
    json_path = os.path.join(pdf_folder, json_name)
    pdf_path = os.path.join(pdf_folder, pdf_name)

    with open(json_path, 'r') as f:
        data = json.load(f)

    if pdf_name not in data:
        raise ValueError(f"{pdf_name} not found in {json_name}.")

    last_index_page = data[pdf_name] - 1  # already 1-based
    index_pages = list(range(last_index_page))  # 0-based

    pattern = re.compile(r'\.*\s*(\d+)\s*$')
    page_numbers = []

    doc = fitz.open(pdf_path)
    for page_num in index_pages:
        page = doc.load_page(page_num)
        text = page.get_text()

        for line in text.split('\n'):
            match = pattern.search(line.strip())
            if match:
                page_number = int(match.group(1))
                page_numbers.append(page_number)

    # Adjust page numbers by adding last_index_page
    page_numbers = [i + last_index_page for i in page_numbers]
    return page_numbers, last_index_page


def split_pdf_by_page_ranges(pdf_path, page_starts, output_folder, base_name):
    """
    Splits a PDF into segments based on start pages, ending each at the next start or EOF.

    Args:
        pdf_path (str): Path to the original PDF.
        page_starts (List[int]): Start pages (0-based, already adjusted).
        output_folder (str): Where to save the output PDFs.
        base_name (str): Prefix for case files.
    """
    os.makedirs(output_folder, exist_ok=True)
    doc = fitz.open(pdf_path)
    total_pages = len(doc)

    # Append end of file for the last range
    page_starts.append(total_pages)

    for i in range(len(page_starts) - 1):
        start = page_starts[i]
        end = page_starts[i + 1]
        new_doc = fitz.open()
        for p in range(start, end):
            new_doc.insert_pdf(doc, from_page=p - 1, to_page=p - 1)
        output_path = os.path.join(output_folder, f"{base_name}-case-{i+1}.pdf")
        new_doc.save(output_path)
        print(f"Saved: {output_path}")


# === Process All PDFs in "trimmed" Folder ===
if __name__ == "__main__":
    folder = "trimmed"
    cases_folder = "cases"
    json_file = os.path.join(folder, "index_mapping.json")

    for file_name in os.listdir(folder):
        if file_name.endswith(".pdf"):
            try:
                print(f"\nProcessing: {file_name}")
                page_list, last_index = extract_index_page_numbers(folder, file_name, debug=True)
                print("Extracted page numbers:", page_list)
                print("Last index page (1-based):", last_index)

                full_pdf_path = os.path.join(folder, file_name)
                base_pdf_name = os.path.splitext(file_name)[0]

                split_pdf_by_page_ranges(full_pdf_path, page_list, cases_folder, base_pdf_name)
            except Exception as e:
                print(f"Error processing {file_name}: {e}")



Processing: 2024-1.pdf
Extracted page numbers: [5, 15, 25, 44, 64, 69, 77, 85, 91, 118, 144, 175, 215, 245, 252, 271, 285, 310, 323, 331, 378, 394, 408, 417, 433, 446, 458, 477, 492, 521, 553, 608, 627, 646, 681, 701, 708, 747, 913, 977, 1049, 1066, 1087, 1094, 1104, 1109, 1132, 1138, 1155, 1169, 1183, 1187, 1189, 1194, 1198, 1223, 1227, 1234, 1239]
Last index page (1-based): 4
Saved: cases\2024-1-case-1.pdf
Saved: cases\2024-1-case-2.pdf
Saved: cases\2024-1-case-3.pdf
Saved: cases\2024-1-case-4.pdf
Saved: cases\2024-1-case-5.pdf
Saved: cases\2024-1-case-6.pdf
Saved: cases\2024-1-case-7.pdf
Saved: cases\2024-1-case-8.pdf
Saved: cases\2024-1-case-9.pdf
Saved: cases\2024-1-case-10.pdf
Saved: cases\2024-1-case-11.pdf
Saved: cases\2024-1-case-12.pdf
Saved: cases\2024-1-case-13.pdf
Saved: cases\2024-1-case-14.pdf
Saved: cases\2024-1-case-15.pdf
Saved: cases\2024-1-case-16.pdf
Saved: cases\2024-1-case-17.pdf
Saved: cases\2024-1-case-18.pdf
Saved: cases\2024-1-case-19.pdf
Saved: cases\2024-1

In [10]:
import fitz  # PyMuPDF
import re

def extract_case_metadata_from_pdf(pdf_path):
    # Open the PDF
    doc = fitz.open(pdf_path)
    first_page = doc.load_page(0)
    text = first_page.get_text()
    doc.close()

    # Clean and split text into lines
    lines = [line.strip() for line in text.strip().split("\n") if line.strip()]

    metadata = {
        "citation": "",
        "case_name": "",
        "appeal_number": "",
        "date": "",
        "judges": ""
    }

    # 1. Citation (first line likely)
    for line in lines[:5]:  # check first 5 lines only to avoid false positives
        if re.search(r"\[\s*(19|20)\d{2}\s*\]", line):
            metadata["citation"] = line
            break

    # 2. Case Name (look for "v.")
    for i in range(1, len(lines) - 1):
        if lines[i].lower() == "v.":
            metadata["case_name"] = f"{lines[i - 1]} v. {lines[i + 1]}"
            break

    # 3. Appeal Number
    for line in lines:
        if "Appeal No" in line:
            metadata["appeal_number"] = line
            break

    # 4. Date (e.g., 02 January 2024)
    for line in lines:
        if re.match(r"\d{2} \w+ \d{4}", line):
            metadata["date"] = line
            break

    # 5. Judges (contains "J." or "JJ.")
    for line in lines:
        if "J." in line or "JJ" in line:
            metadata["judges"] = line.replace("[", "").replace("]", "")
            break

    return metadata

pdf_path = "cases/2024-1-case-2.pdf"
metadata = extract_case_metadata_from_pdf(pdf_path)
print(metadata)


{'citation': '[2024] 1 S.C.R. 11 : 2024 INSC 8', 'case_name': 'Mary Pushpam v. Telvi Curusumary & Ors.', 'appeal_number': '(Civil Appeal No. 9941 of 2016)', 'date': '03 January 2024', 'judges': 'Vikram Nath* and Rajesh Bindal, JJ.'}


In [10]:
import fitz  # PyMuPDF
import re
from collections import defaultdict

# Case-sensitive headings (must match exactly as they appear in the document)
LEGAL_HEADINGS = [
    "Issue for Consideration",
    "Headnotes",
    "List of Citations and Other References",
    "List of Acts",
    "List of Keywords",
    "Other Case Details Including Impunged Order and Appearances",
    "Judgment"
]

def extract_text_from_pdf(pdf_path):
    """Extract all text from PDF."""
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text() + "\n"
    return text

def normalize_text(text):
    """
    Normalize exact headings only (case-sensitive, optional colons).
    Converts 'List of Acts:' or 'List of Acts :' → '\nList of Acts\n'
    """
    for h in LEGAL_HEADINGS:
        # Match exact heading with optional colon and possible trailing spaces
        pattern = re.compile(fr'^\s*{re.escape(h)}\s*:?\s*$', re.MULTILINE)
        matches = pattern.findall(text)
        for match in matches:
            text = text.replace(match, f'\n{h}\n')
    return text

def split_by_headings(text, headings):
    """
    Split text using headings, collect all parts per heading (case-sensitive).
    """
    heading_pattern = '|'.join([fr'\n({re.escape(h)})\n' for h in headings])
    parts = re.split(heading_pattern, text)

    result = defaultdict(list)
    i = 1
    while i + 1 < len(parts):
        heading = parts[i]
        content = parts[i + 1]
        if heading and content:
            result[heading.strip()].append(content.strip())
        i += 2

    return dict(result)

def extract_legal_sections(pdf_path):
    """Full pipeline to extract all case sections (case-sensitive)."""
    full_text = extract_text_from_pdf(pdf_path)
    normalized_text = normalize_text(full_text)

    print("\n🔍 Exact-Match Heading Candidates Found:")
    for line in normalized_text.splitlines():
        if line.strip() in LEGAL_HEADINGS:
            print(f"> {line.strip()}")

    return split_by_headings(normalized_text, LEGAL_HEADINGS)


# 🧪 Example usage
if __name__ == "__main__":
    pdf_path = "cases/2024-1-case-3.pdf"  # change this path
    sections = extract_legal_sections(pdf_path)

    print("\n🧾 Extracted Sections:")
    for heading, blocks in sections.items():
        print(f"\n--- {heading} ---")
        for i, block in enumerate(blocks, 1):
            print(f"\n🔹 Part {i}:\n{block[:1000]}\n")



🔍 Exact-Match Heading Candidates Found:
> Issue for Consideration
> Headnotes
> List of Citations and Other References
> List of Keywords
> Judgment
> Judgment
> Judgment
> Headnotes

🧾 Extracted Sections:

--- Judgment ---

🔹 Part 1:
and Order dated 15.09.2021 of the High Court of 
Judicature at Allahabad in Special Appeal Nos.1435 and 1445 of 2023]

[2024] 1 S.C.R.
23
RADHEY SHYAM YADAV & ANR. ETC. v. STATE OF U.P. & ORS.
Appearances:
Surender Kumar Gupta, Chitvan Singhal, Advs. for the Appellants.
Ms. Sansriti Pathak, Krishnanand Pandeya, Dhawal Uniyal, Naresh 
Kumar, Himanshu Sharma, Advs. for the Respondents.


🔹 Part 2:
/ Order of The Supreme Court


🔹 Part 3:
K.V. Viswanathan, J.
1.	
Leave granted.
2.	
Radhey Shyam Yadav, Lal Chandra Kharwar and Ravindra Nath 
Yadav are the three appellants. On 25.06.1999, they were appointed 
as Assistant Teachers at the Junior High School, Bahorikpur, 
Maharajganj, District Jaunpur, U.P. (hereinafter referred to as ‘the 
School’). From Octobe

In [11]:
import fitz  # PyMuPDF

def read_pdf_text(pdf_path):
    """Reads and returns all text from the PDF at the given path."""
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    return full_text

# 🧪 Example usage
if __name__ == "__main__":
    pdf_path = "cases/2024-1-case-3.pdf"  # replace with your actual path
    text = read_pdf_text(pdf_path)
    print(text)  # print full PDF text


* Author
[2024] 1 S.C.R. 21 : 2024 INSC 7
Case Details
Radhey Shyam Yadav & Anr. Etc.
v.
State of U.P. & Ors.
(Civil Appeal Nos.20-21 Of 2024)
03 January 2024
[J.K. Maheshwari and K.V. Viswanathan*, JJ.]
Issue for Consideration
Three appellants herein were appointed as Assistant Teachers 
at the Junior High School on 25.06.1999. From October, 2005, 
abruptly their salaries were stopped. Whether the State was justified 
in abruptly stopping their salary.
Headnotes
Service Law – Recruitment – Stoppage of salary – The 
District Basic Education Officer case was that by order dated 
26.12.1997, only two additional posts of Assistant Teacher were 
created by the Joint Director of Education – It was averred that 
manipulation was made by the management in collusion with 
the appellants to show that three posts of Assistant Teacher 
were sanctioned – From October, 2005, abruptly salaries of 
appellants were stopped – Propriety:
Held: Apart from the bare allegation, absolutely no material was 
